# From the coursework to a model factory — making training fast, everywhere, for CNNs *and* Transformers

*A CSED 504 journey notebook. Runnable end-to-end; every demo is deliberately short so you can
**watch** it. The full studies it references (`cifar10_train.ipynb`, `../a1-imagenet32/`, `../perf/`)
are the heavy runs — this notebook is the story of how we got them fast.*

---

## Where the coursework left us

Three courses, and the through-line is **how much of the training you write yourself**:

- **CSED 501 — classical ML in `sklearn` / `pandas`.** Regression, classification, clustering (with a
  hand-coded **K-means / K-means++**), and a first `sklearn` neural net + tabular RL (`assignment4`).
  Here **`model.fit()` *is* the training** — you never see a loop, a gradient, or a device. This is
  where the foundations live: bias/variance, regularization (Ridge), kernels (SVM), probability.
- **CSED 502 — drop to the metal.** Backprop, a hand-written **`Solver`** (`csed502/.../solver.py`),
  BatchNorm and ConvNets from scratch, *then* the PyTorch intro (`csed502/assignment4/PyTorch.ipynb`)
  and hand-built attention for **captioning** (`assignment5`). Now **you** write the loop.
- **CSED 503 — industrialize it.** Linear text models, n-grams, embeddings, **self-attention from
  scratch** (`csed503/assignment4`, minGPT-style), then *using* GPT-2/Qwen. The loop gets handed to
  HuggingFace's `Trainer`, and the first **cross-platform sweep** appears.

So the ladder is: **`sklearn.fit()`  →  write the loop by hand  →  industrialize it.** Two rungs were
only *just* reached by the end — and they're exactly what a model factory needs:

1. **GPU training.** First appears in `502/a4 PyTorch.ipynb` as a `USE_GPU` flag + `.to(device)` — and
   that's *it*. No mixed precision, no multi-GPU, no data-pipeline tuning. 503 got as far as
   `Trainer(fp16=True)` + `DataParallel`, but **we never wrote AMP ourselves.**
2. **Vision Transformers.** We hand-built attention for *sequences* (captions, language). A **ViT** —
   attention over image patches — we never touched.

And 503 already planted the **factory seed**: `csed503/assignment1/hyper_sweep.py` (a cross-platform
hyperparameter sweep that auto-picks CUDA+`torch.compile` / MPS / CPU-multiprocessing) and
`csed503/cuda_check.py` (multi-GPU selection, MPS, an FP16 pre-flight). **That `cuda_check.py` is the
direct ancestor of this repo's `src/common/gpu_check.py`.**

This notebook adds the next rung: turn "`.to(device)` and pray" into training that is **fast across
Mac / Windows / Linux / Colab, for both CNNs and Transformers** — the backbone a model factory runs on.

## 1. From `sklearn.fit()` to a `Solver` to the training loop

In **501** you called `model.fit()` and never saw the loop. In **502** you *wrote* it — `Solver._step()`
by hand. In PyTorch it's that same loop, one rung up — and it's the one piece PyTorch still makes you
write yourself, so it's where everything that makes training *fast* has to go. The idiomatic-PyTorch
mapping is worked through cell-by-cell in **[`cifar10_train.ipynb`](cifar10_train.ipynb)**; the one-liner:

| CSED 502, by hand | PyTorch |
|---|---|
| `softmax_loss` (`layers.py`) | `nn.CrossEntropyLoss` |
| `ThreeLayerConvNet` (`cnn.py`) | `torchvision.models.resnet18` |
| `optim.sgd` / `adam` (`optim.py`) | `torch.optim.SGD` / `AdamW` |
| `Solver._step()` (`solver.py`) | the `for x, y in loader:` loop below |
| `Solver._save_checkpoint()` | `torch.save(model.state_dict())` |

Everything in that table is just the coursework with faster, less-buggy parts swapped in. **The rest of
this notebook is what the coursework never reached: making that loop *fast*, and *portable*.**

In [ ]:
# -- Setup: ../common = gpu_check (grown-up cuda_check.py), ../a1-imagenet32 = models.py, ../perf --
import os, sys, time
for rel in ('../common', '../a1-imagenet32', '../perf'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
import torch.nn as nn
from gpu_check import get_device, set_seed, enable_fast_matmul
import models as M
set_seed(42)

## 2. One device, four platforms

In 502 the GPU cell was three lines: a flag, `torch.device('cuda:0')`, `.to(device)`. That breaks the
moment someone on the team opens the notebook on a **Mac** (no CUDA — Apple's **MPS** backend), or on
**Colab** (a different GPU each time), or on a **CPU-only laptop**.

`get_device()` is what `csed503/cuda_check.py` grew into: it prefers **CUDA → MPS → CPU**, smoke-tests
MPS (some macOS versions advertise it but crash on the first op), and on a multi-GPU box selects the
best same-architecture set. It runs unchanged on all four platforms — the same notebook, everywhere.

In [ ]:
DEVICE = get_device()          # CUDA on this box; MPS on a Mac; CPU otherwise
if DEVICE.type == 'cuda':
    enable_fast_matmul()       # TF32 + cuDNN autotune (no-op off CUDA)
print('\nTraining on:', DEVICE)

## 3. Making the CNN fast — the part we worked out together

Here's where the notebook stops being 502-with-nicer-names. Four things move the needle, **none of
them in the coursework**, each one a small collaboration between "try this" and "measure it":

- **Mixed precision (AMP).** 502/503 never hand-wrote AMP (503 only got it via HF's `Trainer(fp16=True)`).
  `torch.autocast` runs the matmuls in **bf16** — ~1.8× on this GPU, and Zen4 CPUs get it too via
  AVX-512-BF16. bf16 needs no loss scaler; fp16 (older CUDA, e.g. a Colab T4) does — the helper picks.
- **Feed the GPU from the GPU.** At 32×32 a small CNN trains faster than CPU workers can hand it data,
  so we keep the whole (tiny) dataset **on the device** and augment there — no DataLoader, no workers.
- **Know *what* the GPU is waiting on.** The single most useful habit we built: is it **CPU-bound**
  (data pipeline), **launch-bound** (batch too small for the card), or **compute-bound** (done)? The
  fix is different for each, and guessing wastes hours.

The demo below is intentionally tiny (a subset, a few epochs) so it finishes while you watch. The full
20-epoch / 92%-accuracy run is `cifar10_train.ipynb`.

In [ ]:
# CIFAR-10 held on the device, augmented on the device (the "feed the GPU from the GPU" trick).
from torchvision.datasets import CIFAR10
from gpu_data import GPUImageLoader          # the loader we built for this; see gpu_data.py
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

_root = os.path.join(os.getcwd(), 'data')
_tr, _te = CIFAR10(_root, train=True, download=True), CIFAR10(_root, train=False, download=True)
def to_device(ds):
    x = torch.from_numpy(ds.data).to(DEVICE).permute(0, 3, 1, 2).contiguous()   # NHWC uint8 -> NCHW
    return x, torch.tensor(ds.targets, device=DEVICE)
xtr, ytr = to_device(_tr); xte, yte = to_device(_te)

DEMO_N, DEMO_EPOCHS = 10_000, 3           # tiny + short so you can watch; scale up for real accuracy
train_loader = GPUImageLoader(xtr[:DEMO_N], ytr[:DEMO_N], 512, MEAN, STD, train=True, seed=42)
test_loader  = GPUImageLoader(xte, yte, 512, MEAN, STD, train=False)
print(f'{DEMO_N:,} train imgs on {DEVICE}, augmented on-device — no DataLoader, no workers')

In [ ]:
# The training loop == Solver._step(), plus AMP.  amp_dtype/scaler chosen for whatever hardware runs this.
from tqdm.auto import tqdm

def pick_amp(device):
    if device.type == 'cuda':
        return (torch.bfloat16, False) if torch.cuda.is_bf16_supported() else (torch.float16, True)
    if device.type == 'cpu':
        return torch.bfloat16, False        # Zen4 AVX-512-BF16; harmless elsewhere
    return None, False                      # MPS: autocast support is patchy — skip

def train_demo(model, epochs, opt, amp_dtype, scaler, mixup=False):
    model = model.to(DEVICE); crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); seen = 0
        for x, y in tqdm(train_loader, desc=f'epoch {ep}/{epochs}', leave=False):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                if mixup:
                    lam = float(np.random.beta(0.2, 0.2)); perm = torch.randperm(x.size(0), device=DEVICE)
                    out = model(lam * x + (1 - lam) * x[perm])
                    loss = lam * crit(out, y) + (1 - lam) * crit(out, y[perm])
                else:
                    loss = crit(model(x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            seen += y.size(0)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        top1 = evaluate(model)
        print(f'epoch {ep}/{epochs}  val top1 {top1:5.1%}  |  {seen/(time.time()-t0):8,.0f} img/s')

@torch.no_grad()
def evaluate(model):
    model.eval(); c = n = 0
    for x, y in test_loader:
        with torch.autocast(DEVICE.type, dtype=amp_dtype_g, enabled=amp_dtype_g is not None):
            c += (model(x).argmax(1) == y).sum().item()
        n += y.size(0)
    return c / n

amp_dtype_g, use_scaler = pick_amp(DEVICE)
print(f'AMP dtype: {amp_dtype_g}  (loss scaler: {use_scaler})')

In [ ]:
# Watch a ResNet-18 train.  ~seconds on a GPU, ~30s on a CPU.
resnet = M.build('resnet18', num_classes=10)
opt = torch.optim.SGD(resnet.parameters(), lr=0.1 * 512 / 256, momentum=0.9, nesterov=True, weight_decay=5e-4)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_scaler)
train_demo(resnet, DEMO_EPOCHS, opt, amp_dtype_g, scaler)

## 4. The Transformer — same job, a recipe of its own

We hand-built attention for *sequences* in class; a **ViT** is that same attention over **image
patches**. The model ports with one argument (`num_classes`) — but the *training recipe* does **not**.

The trap (worked through in `cifar10_train.ipynb` §9): hand the ViT the ResNet's recipe — SGD, `lr=0.1`,
crop+flip — and it lands in the 50s–60s. That's not the architecture failing; it's **a transformer
trained like a CNN**. The fixes: **AdamW** (transformers are trained with Adam ~universally), LR
**warmup**, and heavier augmentation (**mixup**). Same model, very different result.

There was a hardware beat here too: on **Apple MPS**, the ViT's backward pass used to crash on a
`.view()` of a non-contiguous tensor — a real "why does this only break on the Mac?" afternoon. The fix
lives in `gpu_check`/the model code; the lesson is that "cross-platform" is something you *debug into
existence*, not a checkbox.

Below: the same short demo, ViT, with the recipe it actually wants.

In [ ]:
# A ViT on the same images, with a transformer recipe (AdamW + warmup + mixup).
vit = M.build('vit', num_classes=10)
opt = torch.optim.AdamW(vit.parameters(), lr=1e-3, weight_decay=0.05)
warm = torch.optim.lr_scheduler.LinearLR(opt, 0.1, total_iters=1)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_scaler)
print(f'ViT: {sum(p.numel() for p in vit.parameters()):,} params (~matched to resnet18)')
train_demo(vit, DEMO_EPOCHS, opt, amp_dtype_g, scaler, mixup=True)
print('\n(A few epochs on 10k images is a pipeline check, not a verdict — the real CNN-vs-ViT\n'
      ' crossover study is cifar10_train.ipynb §9 and ../a1-imagenet32/.)')

## 5. Predict the cost *before* you pay it

`hyper_sweep.py` (503 A1) searched hyperparameters by **running** them. A factory can't afford that at
scale — a bad config can be a 2-day run. So `../perf/` adds the piece the coursework didn't: **estimate
wall-clock before training**, from the machine's real ceilings and the workload's FLOPs. It's the
*redesign gate* — "2 minutes or 2 days?" answered in seconds.

In [ ]:
import perfkit as pk

fp = pk.fingerprint(DEVICE)
print(f"machine: {fp.get('gpu', fp['device_type'])}  ({fp['device_type']})")

# A quick burst-GEMM probe (seconds, not the full 90s sustained probe) -> peak TFLOPS for the roofline.
peak = pk.probe_matmul_tflops(DEVICE)
p_tflops = peak.get('bf16') or peak.get('tf32') or peak.get('fp32')

flops = pk.count_flops_per_image(lambda: M.build('resnet18', num_classes=10))
work = pk.workload_spec('resnet18-cifar10', n_train=50_000, n_val=10_000,
                        batch_size=512, epochs=20, flops=flops)
est = pk.tier1_estimate(work, p_tflops=p_tflops)     # instant analytical roofline (no run needed)
# The band edges: 'optimistic' = high GPU efficiency (fast), 'pessimistic' = low (slow).
print(f"peak ~{p_tflops:.0f} TFLOPS  |  resnet18 20-epoch estimate: "
      f"{est['optimistic']['total_human']} – {est['pessimistic']['total_human']}  "
      f"(bound by {est['pessimistic']['binding_term']})")

In [ ]:
# The factory's memory: every machine that has run collect.py leaves a record. Predictions compound.
recs = pk.load_records()
if recs:
    print(f"{'machine':34s} {'workload':22s} {'t_step':>8s} {'predicted':>10s} {'actual':>9s}")
    for r in recs:
        f, c, p = r['fingerprint'], r.get('calibration') or {}, r.get('prediction') or {}
        print(f"{(f.get('gpu') or f['device_type'])[:33]:34s} {r['workload']['name'][:21]:22s} "
              f"{(c.get('t_step_s') or 0)*1e3:6.1f}ms {p.get('total_human','-'):>10s} "
              f"{pk.fmt_duration(r['actual_total_s']) if r.get('actual_total_s') else '-':>9s}")
else:
    print('No records yet — run  python ../perf/collect.py  on this machine to add one.')

## 6. The model factory

Put the pieces together and you have the backbone the coursework's `hyper_sweep.py` was reaching for:

- **runs anywhere** — one `get_device()`, four platforms (§2);
- **fast** — AMP + on-device data + the CPU/launch/compute-bound diagnostic (§3);
- **both families** — CNN *and* Transformer, each with its own recipe (§4);
- **predictable** — cost estimated before the run, so search replaces babysitting (§5).

That is a **model factory**: point it at a search space (architecture family, width/depth, recipe),
let it estimate-then-schedule runs across whatever hardware is on hand — the two RTX PRO 6000s here, a
Mac overnight, a Colab tab — and keep the accuracy/cost Pareto front. The *what to search* is the
CNN-vs-Transformer-vs-data-scale study (`../README.md`, `../a1-imagenet32/`, `cifar100_hf_train.ipynb`);
the *how to run it cheaply, everywhere* is everything above.

We started at `.to(device)` and a hand-written `Solver`. This is where that goes.